# 03 — Registre sensibilité / RGPD

Toute colonne est cataloguée. Une variable sensible peut entrer dans l'IA **si étiquetée et justifiée**. Le nominatif reste dans le coffre.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [2]:
from src.data.paths import ProjectPaths, get_project_root
from src.data.registre import load_registre, build_source_column_index, assert_registre_covers_sources, assert_justifications
from src.data.load import load_csv
from src.data.identity import split_identity_and_pseudonymise
from src.data.cohort import filter_domicile
import pandas as pd

paths = ProjectPaths(root=get_project_root())
registre = load_registre(paths.registres / "registre_colonnes.csv")
assert_registre_covers_sources(registre, build_source_column_index(paths))
assert_justifications(registre)
registre.groupby(["sensibilite", "usage_score_sortie"]).size()

sensibilite       usage_score_sortie
aucune            exclu                  1
faible            exclu                 21
identite_directe  exclu                  2
proxy_socio       autorise               3
                  flag_only              2
sante_art9        autorise              46
                  exclu                  4
                  flag_only              1
dtype: int64

In [3]:
registre.loc[registre["sensibilite"]=="identite_directe"]

,fichier_source,colonne,grain,categorie,sensibilite,usage_score_sortie,usage_score_tele,justification_hopital,risque_principal,regle_nettoyage
1,patients.csv,NomPrenom,patient,identite,identite_directe,exclu,exclu,NaN,secret_medical,coffre identité ; jamais dans curated
8,patients.csv,PersonneAPrevenir,patient,identite,identite_directe,exclu,exclu,NaN,secret_medical,coffre identité ; jamais dans curated


In [4]:
patients, _ = split_identity_and_pseudonymise(load_csv(paths, "patients.csv", from_raw=False))
# patients ici = vault; on recharge pseudo
from src.data.identity import split_identity_and_pseudonymise as sp
vault, pseudo = sp(load_csv(paths, "patients.csv", from_raw=False))
sejours = filter_domicile(load_csv(paths, "sejours.csv", from_raw=False))
merged = sejours.merge(pseudo, on="PatientID", how="left")
print("n Domicile", len(merged), "sexe:\n", merged["Sexe"].value_counts(dropna=False))
assert "NomPrenom" not in merged.columns
print("OK registre")

n Domicile 480 sexe:
 Sexe
F                244
M                216
Non renseigné     20
Name: count, dtype: int64
OK registre
